# 🚀 Laya Decision Engine - Kaggle Server (CPU Only + All 3 Checkpoints Preloaded)

Notebook chạy **Laya Server** trên **CPU Kaggle** (tận dụng 30 GB RAM để nạp sẵn toàn bộ 3 checkpoints: English, Multilingual, Typed-Decisions) và public qua **ngrok** với static domain cố định.
Cài đặt trực tiếp từ fork repository [binhminhanh1235/laya-s1-decision](https://github.com/binhminhanh1235/laya-s1-decision).

### ⚠️ Yêu cầu trước khi chạy:
1. **Bật Internet**: Tại thanh bên phải Notebook -> chọn **Internet On**.
2. **Thêm Secret**: Menu trên cùng -> Add-ons -> Secrets -> thêm secret với Label `NGROK_AUTHTOKEN` và Value là auth token Ngrok của bạn.
3. **Accelerator**: Chọn **None** (CPU) để không tốn GPU quota 30h/tuần của Kaggle.


In [ ]:
# =============================================================================
# 🚀 1 CELL DUY NHẤT: CÀI ĐẶT, KẾT NỐI NGROK & CHẠY LAYA SERVER (3 CHECKPOINTS TRÊN CPU)
# =============================================================================
import os
from kaggle_secrets import UserSecretsClient

# 1. Cài đặt pyngrok và laya kèm serve & mcp
!pip install -q pyngrok "git+https://github.com/binhminhanh1235/laya-s1-decision.git#egg=laya[serve,mcp]"

from pyngrok import ngrok

# 2. Lấy Ngrok Token từ Kaggle Secrets & cấu hình
user_secrets = UserSecretsClient()
ngrok_token = user_secrets.get_secret("NGROK_AUTHTOKEN")
ngrok.set_auth_token(ngrok_token)

# Đóng tunnel cũ (nếu có khi rerun lại cell) để tránh xung đột domain tĩnh
ngrok.kill()

# 3. Mở tunnel ngrok với domain cố định
public_url = ngrok.connect(8000, domain="ferret-above-formerly.ngrok-free.app")
separator = "=" * 55
print(f"\n{separator}")
print(f"👉 Public Domain:    {public_url}")
print(f"👉 Health Check:     {public_url}/health")
print(f"👉 Inference API:    {public_url}/v1/systemone")
print(f"👉 MCP SSE Endpoint:  {public_url}/sse")
print(f"👉 Device:           CPU (Kaggle {os.cpu_count() or 4} cores, 30 GB RAM)")
print("👉 Loaded Models:    english, multilingual, typed-decisions (Preloaded)")
print("👉 Auto-task route:  Enabled (Coding/Dev tasks -> typed-decisions)")
print(f"{separator}\n")

# 4. CẤU HÌNH NẠP CẢ 3 CHECKPOINTS TRÊN CPU (TẬN DỤNG 30 GB RAM)
os.environ["CUDA_VISIBLE_DEVICES"] = ""             # Vô hiệu hóa GPU, không cấp phát VRAM
os.environ["LAYA_DEVICE"] = "cpu"                   # Bắt buộc Laya chạy inference trên CPU
os.environ["LAYA_PRELOAD"] = "1"                    # Nạp sẵn cả 3 model vào RAM lúc khởi động
os.environ["LAYA_MODELS"] = "english,multilingual,typed-decisions" # Nạp đủ cả 3 checkpoints
os.environ["LAYA_MAX_LOADED"] = "3"                # Giữ đồng thời cả 3 model trong 30 GB RAM
os.environ["LAYA_AUTO_TASK"] = "1"                 # Tự động route tác vụ lập trình sang typed-decisions
os.environ["LAYA_THREADS"] = str(os.cpu_count() or 4) # Tối ưu đa luồng CPU (intra-op threads)
os.environ["LAYA_PORT"] = "8000"

# 5. Khởi động Laya server trên port 8000
!python -m laya.serve
